In [1]:
import os
import pandas as pd
import numpy as np
from Bio import SeqIO
from Bio.Seq import Seq
# --------------------------------
from scipy.stats import mannwhitneyu, ttest_ind, pearsonr
from scipy.cluster.hierarchy import linkage, leaves_list
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from statsmodels.formula.api import ols
# --------------------------------
from tqdm.notebook import tqdm
# ----------------------------------------------------------------
from lets_plot import *
LetsPlot.setup_html()
base_size = 14
tsne_plot_width = 650
theme_settings = theme(
    axis_text=element_text(size=base_size),
    axis_title=element_text(size=base_size * 1.2),
    legend_title=element_text(size=base_size * 1.2),
    legend_text=element_text(size=base_size * 0.9),
    plot_title=element_text(size=base_size * 1.4),
    axis_ticks_y=element_line(color='black', size=0.5),
    axis_line_y=element_line(color='black', size=0.5)
)

In [2]:
## function to generate scatter plot colored by a continuous variable

import matplotlib.pyplot as plt
import seaborn as sns

def plot_scatter_colored(tab_data, x_name, y_name, color_col,
                         x_lab=None, y_lab=None,
                         cmap='viridis', fig_size=(8, 8),
                         point_size=10, alpha=0.8,
                         vmin=None, vmax=None,
                         file_name=None, dpi=300, 
                         show_plot=True):
    """
    Plot a simple scatter plot colored by a continuous variable, with optional clipping.

    Parameters
    ----------
    tab_data : pandas.DataFrame
        Input table containing x, y, and color columns.
    x_name, y_name, color_col : str
        Column names for x-axis, y-axis, and color mapping.
    cmap : str
        Matplotlib colormap name for the continuous variable.
    vmin, vmax : float or None
        Minimum and maximum values to clip the color variable.
    """
    plt.figure(figsize=fig_size)

    # Clip color values if requested
    colors = tab_data[color_col].copy()
    if vmin is not None:
        colors = np.maximum(colors, vmin)
    if vmax is not None:
        colors = np.minimum(colors, vmax)

    sc = plt.scatter(
        tab_data[x_name],
        tab_data[y_name],
        c=colors,
        cmap=cmap,
        s=point_size,
        alpha=alpha,
        edgecolors='none',
        rasterized=True,
        vmin=vmin,
        vmax=vmax
    )

    plt.colorbar(sc, label=color_col)

    plt.xlabel(x_lab if x_lab else x_name, fontsize=10)
    plt.ylabel(y_lab if y_lab else y_name, fontsize=10)

    if file_name:
        plt.savefig(file_name, format='pdf', bbox_inches='tight', dpi=dpi)
    if show_plot:
        plt.show()
    else:
        plt.close()

In [29]:
## paths & load data
dir_data = os.path.join('resubmission', 'data')
dir_results = os.path.join('resubmission', 'results')
dir_plotting = os.path.join(dir_results, 'plots')
dir_models = os.path.join(dir_results, 'models')
path_metadata = os.path.join(dir_data, 'metadata_selected_encodespeckleRBPs.csv')  #metadata_selected_encodespeckleRBPs.csv

tab_meta = pd.read_csv(path_metadata, index_col=0)
high_cut, low_cut = -0.364742, -1.342584 # Hl class boundaries

## shared ENCORE inputs
tab_pks_enc = pd.read_csv(os.path.join(dir_data, 'external', 'ENCORE', 'peaks_encore.csv'))
tab_pks_enc = tab_pks_enc[~tab_pks_enc['intron'].duplicated(keep=False)]
tab_pks_enc.index = tab_pks_enc['intron']
tab_pks_enc = tab_pks_enc.drop(columns=['intron'])
tab_pks_enc.index.name = 'EVENT'

tab_speckle_rbps = pd.read_csv(
    os.path.join(dir_data, 'external', 'ENCORE', 'HepG2-HeLa_FeaturesMatrix.csv')
)
rbp_speckle = tab_speckle_rbps.query('Quality > 1').dropna(subset=['Speckles'])['RBP'].to_list()

In [30]:
# Supplementary Figure 4 - Correlation plot

variables = ['Repeat_Percentage', 'phylop_mean', 'LENGTH', 'GC_Content', 
             'Nuclear_Enrichment', 'Speckle_Enrichment', 'CpG_methylated_perc', 'ssDRIP_cov',
             'PIR_Nuc_baseline', 'hl_gated_intwise_scaled_logged', 'n_specklerbp_peaks_sum_lnorm_logged']

n_vars = len(variables)
corr_matrix = np.zeros((n_vars, n_vars))
p_matrix = np.zeros((n_vars, n_vars))
n_matrix = np.zeros((n_vars, n_vars))
for i, var1 in enumerate(variables):
    for j, var2 in enumerate(variables):
        mask = ~(tab_meta[var1].isna() | tab_meta[var2].isna())
        if mask.sum() > 1:
            vvar1 = tab_meta.loc[mask, var1]
            vvar2 = tab_meta.loc[mask, var2]
            if var1 == 'LENGTH':
                vvar1 = np.log10(vvar1)
            elif var1 == 'Repeat_Percentage':
                vvar1 = vvar1.clip(upper=100)
            if var2 == 'LENGTH':
                vvar2 = np.log10(vvar2)
            elif var2 == 'Repeat_Percentage':
                vvar2 = vvar2.clip(upper=100)
            corr, p_val = pearsonr(vvar1, vvar2)
            corr_matrix[i, j] = corr
            p_matrix[i, j] = p_val
            n_matrix[i, j] = mask.sum()   # <---- record N
        else:
            corr_matrix[i, j] = np.nan
            p_matrix[i, j] = np.nan
            n_matrix[i, j] = np.nan
##
corr_df = pd.DataFrame(corr_matrix, index=variables, columns=variables)
p_df = pd.DataFrame(p_matrix, index=variables, columns=variables)
n_df  = pd.DataFrame(n_matrix, index=variables, columns=variables)
##
distance_matrix = 1 - np.abs(corr_df.values)
linkage_matrix = linkage(distance_matrix, method='ward')
clustered_order = leaves_list(linkage_matrix)
ordered_variables = [variables[i] for i in clustered_order]
##
corr_melted = corr_df.reset_index().melt(
    id_vars='index', 
    var_name='variable2', 
    value_name='R'
).rename(columns={'index': 'variable1'})
p_melted = p_df.reset_index().melt(
    id_vars='index', 
    var_name='variable2', 
    value_name='p_val'
).rename(columns={'index': 'variable1'})
n_melted = n_df.reset_index().melt(
    id_vars='index', 
    var_name='variable2', 
    value_name='N'
).rename(columns={'index': 'variable1'})


corr_melted['variable1'] = pd.Categorical(corr_melted['variable1'], 
                                          categories=ordered_variables, 
                                          ordered=True)
corr_melted['variable2'] = pd.Categorical(corr_melted['variable2'],
                                          categories=ordered_variables, 
                                          ordered=True)
## report p-values
corr_reporting = corr_melted.merge(
    p_melted, on=['variable1', 'variable2']
).merge(
    n_melted, on=['variable1', 'variable2']
)
## tidy up
corr_reporting.columns = ['X', 'Y', 'Pearson_R', 'p_value', 'N']
corr_reporting['p_adj'] = multipletests(corr_reporting.p_value, method='fdr_bh')[1]
corr_reporting = corr_reporting.query('X != Y')
corr_reporting.to_csv(os.path.join(dir_plotting, 'SF4A_tiles_cors_stats.csv'), index=False)

# Source data export for Supplementary Fig4A heatmap
corr_melted.to_csv(os.path.join(dir_plotting, 'SF4A_tiles_cors_all_source.csv'), index=False)

## triangular for the heatmap? ---> looks ugly
# var_to_idx = {var: i for i, var in enumerate(ordered_variables)}
# corr_melted['idx1'] = corr_melted['variable1'].map(var_to_idx)
# corr_melted['idx2'] = corr_melted['variable2'].map(var_to_idx)
# corr_triangular = corr_melted[corr_melted['idx1'] >= corr_melted['idx2']].copy()

gg_tiles_cors_all = ggplot(data=corr_melted) +\
    geom_tile(aes(x='variable1', y='variable2', fill='R'), size=1)+ \
    scale_fill_gradient2(low='#a76794', mid='white', high='#2a1f3f') +\
    theme_settings +\
    ggsize(650, 600)
ggsave(gg_tiles_cors_all, filename='SF4A_tiles_cors_all.svg', path=dir_plotting)

/home/icb/artem.baranovskii/tmp/ipykernel_3800196/2571863335.py:39: ClusterWarning: The symmetric non-negative hollow observation matrix looks suspiciously like an uncondensed distance matrix
  linkage_matrix = linkage(distance_matrix, method='ward')


'/ictstr01/groups/crna01/projects/collabs/gabi/resubmission/results/plots/SF4A_tiles_cors_all.svg'

In [22]:
tmp_scat = tab_meta.copy()
## add a logged-length column for plotting
tmp_scat["LENGTH_logged"] = tmp_scat["LENGTH"].apply(lambda x: np.log10(x) if pd.notnull(x) else np.nan)
## append HL classes
class_conds = [
    tmp_scat['hl_gated_intwise_scaled_logged'] > high_cut,
    tmp_scat['hl_gated_intwise_scaled_logged'] < low_cut
]
class_names = ['LL-RIs', 'SL-RIs']
tmp_scat['class_hls'] = np.select(class_conds, class_names, default='else')

## append IR classes
class_conds = [
    tmp_scat['PIR_Nuc_baseline'] >= 30,
    tmp_scat['PIR_Nuc_baseline'] < 1
]
class_names = ['Retained', 'Spliced']
tmp_scat['class_ir'] = np.select(class_conds, class_names, default='else')

## append umap reductions' coordinates
tab_embeds_hls = pd.read_csv(os.path.join(dir_models, 'HL_revised/parnet_512_jc_50percgap_256bp_PIR10/umap_embeddings.csv'), index_col=0)
tab_embeds_hls.index.name = 'EVENT'
tab_embeds_ir = pd.read_csv(os.path.join(dir_models, 'IR/umap_embeddings.csv'), index_col=0)
tab_embeds_ir.index.name = 'EVENT'

tmp_scat = tmp_scat.merge(
    tab_embeds_hls, left_index=True, right_index=True, how='left'
).merge(
    tab_embeds_ir, left_index=True, right_index=True, how='left'
)

## bound dictionary for scatter plot color limits
color_bounds = {
    'phylop_mean': (-1, 1),
    'GC_Content': (0, 80),
    'Repeat_Percentage': (0, 100),
    'Speckle_Enrichment': (-2, 2),
    'Nuclear_Enrichment': (-1.5, 1.5),
    'ssDRIP_cov': (0, 2),
    'CpG_methylated_perc': (0, 100),
    'LENGTH_logged': (2, 5),
    'PIR_Nuc_baseline': (0, 60)
}

# Unified UMAP source table for SF4 scatter exports.
umap_color_vars = []
for varbl in variables:
    if varbl == 'hl_gated_intwise_scaled_logged':
        continue
    out_var = 'LENGTH_logged' if varbl == 'LENGTH' else varbl
    if out_var not in umap_color_vars:
        umap_color_vars.append(out_var)

umap_source_cols = ['UMAP1_IR', 'UMAP2_IR', 'UMAP1_HLrev', 'UMAP2_HLrev'] + umap_color_vars
umap_source_cols = [c for c in umap_source_cols if c in tmp_scat.columns]

tab_umap_source = tmp_scat[umap_source_cols].copy()
tab_umap_source.index.name = 'EVENT'
tab_umap_source.reset_index().to_csv(os.path.join(dir_plotting, 'SF4_umap_corscatters_source.csv'), index=False)

In [23]:
if not os.path.exists(os.path.join(dir_plotting, 'SF4_corscatters')):
    os.mkdir(os.path.join(dir_plotting, 'SF4_corscatters'))
for varbl in variables:
    if varbl == 'LENGTH':
        varbl = 'LENGTH_logged'
    if varbl == 'hl_gated_intwise_scaled_logged':
        continue
    plot_scatter_colored(
        tab_data=tmp_scat[tmp_scat.class_ir.isin(['Spliced', 'Retained'])],
        x_name='UMAP1_IR', y_name='UMAP2_IR',
        color_col=varbl, vmin=color_bounds.get(varbl, None)[0], vmax=color_bounds.get(varbl, None)[1],
        cmap=sns.cubehelix_palette(as_cmap=True, rot=.4, start=-.05),
        point_size=7, alpha=0.55, fig_size=(8,7),
        show_plot=False, file_name=os.path.join(dir_plotting, 'SF4_corscatters', f'{varbl}_vs_IR.pdf')
    )
    plot_scatter_colored(
        tab_data=tmp_scat[tmp_scat.class_hls.isin(['LL-RIs', 'SL-RIs'])],
        x_name='UMAP1_HLrev', y_name='UMAP2_HLrev',
        color_col=varbl, vmin=color_bounds.get(varbl, None)[0], vmax=color_bounds.get(varbl, None)[1],
        cmap=sns.cubehelix_palette(as_cmap=True, rot=.4, start=-.05),
        point_size=15, alpha=0.65, fig_size=(8,7),
        show_plot=False, file_name=os.path.join(dir_plotting, 'SF4_corscatters', f'{varbl}_vs_HLS.pdf')
    )

In [ ]:
## Interesting question - what is the order of intron in the transcript?
path_vastdb = os.path.join(dir_data, 'external', 'EVENT_INFO-hg38.tab')
if not os.path.exists(path_vastdb):
    raise FileNotFoundError(
        f"VASTDB annotation file not found at {path_vastdb}. Please get one at https://vastdb.crg.eu/downloads/hg38/EVENT_INFO-hg38.tab.gz."
    )
vast_annot = pd.read_csv(path_vastdb, sep='\t')[lambda df: df['EVENT'].str.contains('HsaINT')]
introns_source = vast_annot[['GENE', 'EVENT', 'COORD_o', 'FULL_CO']].assign(
    chrom = vast_annot['COORD_o'].str.split(':').str[0],
    start = vast_annot['COORD_o'].str.split(':').str[1].str.split('-').str[0].astype(int),
    end = vast_annot['COORD_o'].str.split(':').str[1].str.split('-').str[1].astype(int),
    strand = vast_annot['FULL_CO'].str.split(':').str[2]
).drop(columns=['COORD_o', 'FULL_CO'], inplace=False).rename(columns={'COORD_o': 'COORD'}).reset_index(drop=True)